![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 17 -- Lab 2: Build Your Own Mini-GPT

**Scenario:** You have heard of ChatGPT, the AI that can write essays, poems, and code. Today, you will build a **tiny version** of the same architecture from scratch! Your Mini-GPT will learn to generate Shakespeare-like text, one token at a time.

This lab connects directly to the **Transformer** architecture from today's slides: self-attention, multi-head attention, positional encoding, and autoregressive generation.

| Part | Goal |
|---|---|
| Part 1 | Load Shakespeare text data |
| Part 2 | BPE tokenization with tiktoken |
| Part 3 | Create training sequences |
| Part 4 | Positional encoding (given) |
| Part 5 | Build the Transformer from scratch |
| Part 6 | Train the model |
| Part 7 | Generate text! |

---

## Setup

In [ ]:
!pip install tiktoken -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import urllib.request

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---

## Part 1: Load Shakespeare Text Data

We download a small file of Shakespeare's works (~1 MB) and take a look at what's inside.

In [ ]:
# --- GIVEN ---
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filepath = 'shakespeare.txt'

import os
if not os.path.exists(filepath):
    print('Downloading Shakespeare...')
    urllib.request.urlretrieve(url, filepath)

with open(filepath, 'r') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'\nFirst 500 characters:')
print('=' * 60)
print(text[:500])
print('=' * 60)

---

## Part 2: BPE Tokenization

Instead of treating each character separately, we use **Byte Pair Encoding (BPE)** -- the same tokenization used by GPT-2 and GPT-3. We use OpenAI's `tiktoken` library which gives us GPT-2's pre-trained tokenizer.

BPE works by:
1. Starting with individual characters
2. Repeatedly merging the most frequent pair of tokens
3. Building up a vocabulary of common subwords

In [ ]:
# --- GIVEN ---
# Load GPT-2's BPE tokenizer
enc = tiktoken.get_encoding('gpt2')

print(f'Vocabulary size: {enc.n_vocab:,}')

# Let's see how it tokenizes text
sample = "To be, or not to be, that is the question"
tokens = enc.encode(sample)
print(f'\nOriginal: "{sample}"')
print(f'Token IDs: {tokens}')
print(f'Number of tokens: {len(tokens)}')
print(f'\nEach token decoded:')
for t in tokens:
    print(f'  {t:>5d} -> "{enc.decode([t])}"')

In [ ]:
# --- GIVEN ---
# Encode the entire Shakespeare text
all_tokens = enc.encode(text)
all_tokens = torch.tensor(all_tokens, dtype=torch.long)

print(f'Total tokens in Shakespeare: {len(all_tokens):,}')
print(f'\nFirst 20 tokens: {all_tokens[:20].tolist()}')
print(f'Decoded back: "{enc.decode(all_tokens[:20].tolist())}"')

VOCAB_SIZE = enc.n_vocab
print(f'\nUsing vocab size: {VOCAB_SIZE:,}')

---

## Part 3: Create Training Sequences

For language modeling, we train the model to predict the **next token** given the previous tokens.

We create training pairs using a **sliding window**:
- Input: tokens at positions `[0, 1, 2, ..., block_size-1]`
- Target: tokens at positions `[1, 2, 3, ..., block_size]` (shifted by 1)

### Task 1: Create the Dataset

**TODO:**
- Complete the `TextDataset` class:
  - `__len__`: return how many sequences we can extract
  - `__getitem__`: return `(x, y)` where `x = tokens[idx : idx + block_size]` and `y = tokens[idx + 1 : idx + block_size + 1]`
- Split into 90% train / 10% val
- Create DataLoaders with `batch_size=32`

In [ ]:
BLOCK_SIZE = 64

class TextDataset(Dataset):
    def __init__(self, tokens, block_size):
        self.tokens = tokens
        self.block_size = block_size

    def __len__(self):
        # Your code here
        pass

    def __getitem__(self, idx):
        # Your code here
        pass

# Split into train (90%) and val (10%)
n = len(all_tokens)
train_tokens = all_tokens[:int(0.9 * n)]
val_tokens = all_tokens[int(0.9 * n):]

train_dataset = TextDataset(train_tokens, BLOCK_SIZE)
val_dataset = TextDataset(val_tokens, BLOCK_SIZE)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Block size: {BLOCK_SIZE}')
print(f'Training sequences: {len(train_dataset):,}')
print(f'Validation sequences: {len(val_dataset):,}')

---

## Part 4: Positional Encoding (GIVEN)

Transformers process all tokens in parallel -- they have no notion of order. **Positional encoding** adds position information so the model knows which token is first, second, third, etc.

We use the **sinusoidal** encoding from the original "Attention Is All You Need" paper. Each position gets a unique pattern of sine and cosine waves at different frequencies -- like a fingerprint for each position.

**You do not need to write this code** -- just run it and look at the heatmap to see how it works.

In [ ]:
# --- GIVEN ---
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention Is All You Need'."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        # Create a matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # position = [0, 1, 2, ..., max_len-1] as a column vector
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # div_term controls the frequency of each sine/cosine wave
        # Higher dimensions = lower frequency = captures longer-range patterns
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Even dimensions get sine, odd dimensions get cosine
        pe[:, 0::2] = torch.sin(position * div_term)  # sin for dimensions 0, 2, 4, ...
        pe[:, 1::2] = torch.cos(position * div_term)  # cos for dimensions 1, 3, 5, ...

        # Register as buffer (not a parameter -- no gradients needed)
        self.register_buffer('pe', pe.unsqueeze(0))  # shape: (1, max_len, d_model)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # Add positional encoding to the input embeddings
        return x + self.pe[:, :x.size(1), :]

# Visualize the positional encoding
pe_vis = PositionalEncoding(d_model=64, max_len=BLOCK_SIZE)
pe_matrix = pe_vis.pe.squeeze(0).numpy()  # (block_size, 64)

plt.figure(figsize=(10, 4))
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu')
plt.colorbar(label='Value')
plt.xlabel('Position in Sequence')
plt.ylabel('Encoding Dimension')
plt.title('Sinusoidal Positional Encoding')
plt.tight_layout()
plt.show()

print('Each column is a unique position fingerprint.')
print('Low dimensions (bottom) oscillate fast -- encode fine position.')
print('High dimensions (top) oscillate slowly -- encode coarse position.')

---

## Part 5: Build the Transformer from Scratch

We will build the Transformer piece by piece, following the slides:
1. **Single Attention Head** -- scaled dot-product attention with causal mask
2. **Multi-Head Attention** -- run multiple heads in parallel
3. **Transformer Block** -- attention + feed-forward + layer norm + residual
4. **MiniGPT** -- the full model: embedding + positional encoding + N blocks + output head

### Task 2: Single Attention Head

From the slides: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

For a **decoder** (like GPT), we also need a **causal mask** so each token can only attend to previous tokens.

**TODO:**
- Create 3 linear layers: `W_Q`, `W_K`, `W_V` (from `d_model` to `head_size`)
- In forward: compute Q, K, V, then scores = Q @ K^T / sqrt(d_k)
- Apply causal mask: `torch.tril(torch.ones(T, T))`, set masked positions to `-inf`
- Softmax the scores, multiply by V

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        # Your code here

    def forward(self, x):
        B, T, C = x.shape  # batch, seq_len, d_model
        # Your code here
        pass

# Quick test
head = AttentionHead(d_model=64, head_size=16)
dummy = torch.randn(2, 10, 64)
out = head(dummy)
print(f'AttentionHead: input {dummy.shape} -> output {out.shape}')

### Task 3: Multi-Head Attention

Run multiple attention heads in parallel, concatenate their outputs, and project back to `d_model`.

**TODO:**
- Create a list of `n_heads` AttentionHead modules (each with `head_size = d_model // n_heads`)
- Create a projection layer: `nn.Linear(d_model, d_model)`
- In forward: run each head, concatenate outputs, project

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # Your code here

    def forward(self, x):
        # Your code here
        pass

# Quick test
mha = MultiHeadAttention(d_model=64, n_heads=4)
out = mha(dummy)
print(f'MultiHeadAttention: input {dummy.shape} -> output {out.shape}')

### Task 4: Transformer Block

One block = Multi-Head Attention + Add & Norm + Feed-Forward + Add & Norm.

**TODO:**
- Create: `MultiHeadAttention`, `LayerNorm`, feed-forward network (Linear -> ReLU -> Linear), another `LayerNorm`, `Dropout`
- In forward: `x = norm1(x + dropout(attention(x)))`, then `x = norm2(x + dropout(ff(x)))`

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        # Your code here

    def forward(self, x):
        # Your code here
        pass

# Quick test
block = TransformerBlock(d_model=64, n_heads=4)
out = block(dummy)
print(f'TransformerBlock: input {dummy.shape} -> output {out.shape}')

### Task 5: The Full MiniGPT Model

Put it all together: token embedding + positional encoding + stack of Transformer blocks + output linear layer.

**TODO:**
- `nn.Embedding(vocab_size, d_model)` for token embeddings
- `PositionalEncoding(d_model, max_len=block_size)` (from Part 4)
- A list of `n_layers` TransformerBlocks
- `nn.LayerNorm(d_model)` for final normalization
- `nn.Linear(d_model, vocab_size)` to predict the next token
- In forward: embed -> add pos encoding -> pass through blocks -> norm -> linear

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        # Your code here

    def forward(self, x):
        # Your code here
        pass

# Model configuration -- small enough for a T4 GPU
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4

model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    block_size=BLOCK_SIZE,
    dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT architecture:')
print(f'  d_model = {D_MODEL}')
print(f'  n_heads = {N_HEADS}')
print(f'  n_layers = {N_LAYERS}')
print(f'  block_size = {BLOCK_SIZE}')
print(f'  vocab_size = {VOCAB_SIZE:,}')
print(f'  Total parameters: {total_params:,}')

---

## Part 6: Train the Model

We train the model to predict the next token at every position. The loss is cross-entropy between the predicted token probabilities and the actual next token.

**TODO:**
- Create an AdamW optimizer with `lr=3e-4`
- For each step: get a batch, forward pass, compute cross-entropy loss, backward, clip gradients, optimizer step
- Print loss every 50 steps

In [ ]:
NUM_STEPS = 500
PRINT_EVERY = 50

# Your code here


In [ ]:
# Plot training loss
# Your code here


---

## Part 7: Generate Text!

Now the fun part. We use **autoregressive generation**: start with a prompt, predict the next token, add it to the sequence, and repeat.

We also add **temperature** control:
- `temperature = 1.0`: normal sampling
- `temperature < 1.0`: more confident/repetitive (sharper distribution)
- `temperature > 1.0`: more random/creative (flatter distribution)

### Task 6: Autoregressive Generation

**TODO:**
- Encode the prompt with `enc.encode(prompt)`
- In a loop for `max_new_tokens` steps:
  - Crop tokens to the last `BLOCK_SIZE` tokens
  - Forward pass through the model
  - Take the logits at the last position only
  - Divide by temperature, apply softmax
  - Sample the next token with `torch.multinomial`
  - Append the new token to the sequence
- Decode everything back to text with `enc.decode`

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=0.8):
    """Generate text autoregressively."""
    model.eval()
    # Your code here
    pass

In [ ]:
# Generate with different prompts
prompts = [
    "To be, or not to be",
    "ROMEO:\n",
    "The king",
]

for prompt in prompts:
    print('=' * 60)
    print(f'PROMPT: "{prompt.strip()}"')
    print('-' * 60)
    output = generate(model, prompt, max_new_tokens=150, temperature=0.8)
    print(output)
    print()

### Task 7: Experiment with Temperature

Try the same prompt with different temperatures to see how it changes the output.

In [ ]:
prompt = "JULIET:\nO Romeo, Romeo!"

for temp in [0.3, 0.8, 1.5]:
    print(f'\n{"=" * 60}')
    print(f'Temperature = {temp}')
    print('-' * 60)
    output = generate(model, prompt, max_new_tokens=100, temperature=temp)
    print(output)

---

## Discussion

1. Does the generated text look like Shakespeare? What does it get right and what does it get wrong?
2. How does temperature affect the quality and diversity of the generated text?
3. Our model has only ~7M parameters. GPT-2 has 124M and GPT-4 has over 1 trillion. How do you think scaling up would improve the results?

---

## Wrap-Up

**What you built today:**
- Used **BPE tokenization** (the same method used by GPT-2/3/4) via tiktoken
- Understood **positional encoding** -- how Transformers know word order
- Built a **Transformer decoder** from scratch: attention head, multi-head attention, transformer block
- Trained it on Shakespeare to do **next-token prediction**
- Used **autoregressive generation** to produce new text
- Experimented with **temperature** to control creativity vs coherence

**Congratulations -- you just built your own (tiny) GPT!**

The exact same architecture (just much bigger) powers ChatGPT, Claude, Gemini, and LLaMA.